[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/37.%20The%20Equipment%20Selection%20Problem%20%28RTG%20vs.%20RMG%20vs.%20Straddle%29/P37-Tier-8_executed.ipynb)

---



# 37. The Equipment Selection Problem (RTG vs. RMG vs. Straddle)
## Tier 8: A Value-Aligned and Ethical Decision Framework

### Goal

This notebook applies a stakeholder-weighted scoring framework to a container-terminal equipment selection problem. It compares configurations of rubber-tired gantry cranes (RTGs), rail-mounted gantry cranes (RMGs), and straddle carriers using economic, operational, environmental, safety, employment, and community attributes.

The method has four stages: define illustrative equipment and stakeholder data; generate a finite list of candidate configurations; reject candidates that violate the budget, productivity, or minimum-compliance rules; and select the remaining candidate with the highest combined score. The final plots and quality indicators describe the selected candidate and a fixed reference configuration.

### Inputs and outputs

The inputs are the equipment attributes and stakeholder preferences defined below, a budget limit, a minimum productivity requirement, and a fixed list of base configurations. The outputs are the selected candidate (if one is feasible), its calculated metrics, and visual summaries of stakeholder utilities and modeled compliance scores.

### Scope and modeling assumptions

- All equipment and stakeholder values are illustrative model inputs, not measured terminal data or validated regulatory standards.
- Equipment units are treated as additive: total capital cost, productivity, and estimated jobs are sums across units. Average attributes, such as environmental impact and safety, are unit-count-weighted averages.
- The model does not represent equipment location, scheduling, interactions, service life, discounting, uncertainty, emissions in physical units, wage quality, or the process by which stakeholders set their preferences.
- Stakeholder priorities that do not have an implemented utility calculation contribute no term to that stakeholder's utility. The code makes this limitation explicit; a score is not a substitute for stakeholder consultation or a legal assessment.
- The candidate generator scales a finite collection of base configurations and converts the scaled unit counts to integers, with a minimum of one unit of each type in every generated configuration. The result is the best feasible candidate in that generated list, not a proof of a global optimum over every possible equipment mix.
- Budget and minimum productivity are feasibility requirements. The ethical scores are not all hard thresholds: a candidate is rejected by the ethical veto only when its lowest compliance score is below 0.3.
- Constitutional principles are represented as a transparent, simplified veto and explanatory framework. The implementation does not constitute a formal constitutional-AI system, establish regulatory compliance, or automatically modify a candidate.

### Method and mathematical model

Let $E$ be the set of equipment types and let $x_e$ be the nonnegative integer number of units of type $e$. For a candidate configuration $x$, total capital cost and productivity are

$$
C(x) = \sum_{e \in E} c_e x_e,
\qquad
P(x) = \sum_{e \in E} p_e x_e,
$$

where $c_e$ and $p_e$ are the capital cost and productivity of equipment type $e$. A candidate is feasible only if $C(x) \leq B$ and $P(x) \geq P_{\min}$, where $B$ is the budget limit and $P_{\min}$ is the minimum productivity.

For stakeholder $s$, the code computes a utility $U_s(x)$ by adding the implemented priority-weighted metric terms. A priority with no implemented metric contributes zero. Stakeholder utilities are combined using stakeholder influence values $a_s$:

$$
U(x) = \sum_{s \in S} \frac{a_s}{\sum_{j \in S} a_j} U_s(x),
$$

where $S$ is the set of stakeholder groups. The function `calculate_stakeholder_utility` calculates each $U_s(x)$, and `calculate_overall_utility` calculates $U(x)$.

Each ethical check returns a modeled score $q_k(x)$ between 0 and 1. The formulas depend on the direction and definition of the indicator: environmental impact is better when lower, while safety, job creation, and community impact are better when higher. The cost-fairness indicator measures balance in equipment counts. For a nonempty configuration, the score used in selection is the weakest indicator score:

$$
q_{\min}(x) = \min_{k \in K} q_k(x),
$$

where $K$ is the set of modeled compliance indicators. Candidates with $q_{\min}(x) < 0.3$ are rejected.

Among the generated candidates that pass the budget, productivity, and minimum-compliance checks, the selection score is

$$
F(x) = w_{\mathrm{economic}} U(x)
      + w_{\mathrm{ethical}} q_{\min}(x)
      + w_{\mathrm{social}} \frac{P(x)}{C(x)}.
$$

The weights are the values set in the optimizer. The productivity-per-cost term is used in its raw scale for candidate selection; the displayed indicator plot separately normalizes that ratio for visualization. Because the terms in $F(x)$ do not all have the same units or scale, the weights do not by themselves make their contributions numerically comparable. This is a constructed weighted score, not a multi-objective Pareto optimization.

The candidate generator evaluates a finite list of scaled base configurations. If $M$ candidates are generated, $|S|$ stakeholder groups and $|K|$ compliance indicators are evaluated for each candidate. The calculation therefore grows with the number of generated candidates and the work needed to evaluate their metrics; it does not search all possible integer configurations. Candidates are considered in list order, and the first candidate is retained when two scores are exactly equal.

### How to read the results

A higher utility or compliance score means a higher value under the notebook's stated scoring rules, not a measured level of stakeholder approval or proof of ethical acceptability. The compliance profile and trade-off plots help identify which modeled criteria contribute to a candidate's evaluation. Conclusions depend on the supplied data, the implemented utility terms, the scoring rules, and the finite candidate-generation procedure.

In [ ]:
import random
import warnings
from dataclasses import dataclass
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore")

np.random.seed(42)
random.seed(42)

print("Equipment Selection: Value-Aligned and Ethical Decision Framework")
print("Stakeholder-weighted scoring over a finite set of candidate configurations")

### Ethical Framework Data Structure

The data classes keep the model inputs, stakeholder preferences, constraint definitions, and evaluated solution together. Cost values are in million USD; operating cost and maintenance are annual million USD values. Productivity is measured in containers per hour. Environmental impact and safety are dimensionless scores on a 0–1 scale, with higher environmental impact being worse and higher safety being better. Job-creation factors are illustrative estimated jobs per equipment unit.

In [ ]:
@dataclass
class EquipmentType:
    name: str
    capital_cost: float  # Million USD
    operating_cost: float  # Million USD per year
    productivity: float  # Containers per hour
    flexibility: float  # 0-1 scale
    maintenance: float  # Million USD per year
    footprint: float  # Square meters
    energy_consumption: float  # kWh per container
    environmental_impact: float  # 0-1 scale; higher is worse
    safety_rating: float  # 0-1 scale; higher is safer
    job_creation_factor: float  # Estimated jobs per unit
    community_impact: float  # 0-1 scale; higher is more beneficial


@dataclass
class Stakeholder:
    name: str
    category: str
    priorities: Dict[str, float]
    influence: float
    utility_function: str
    ethical_constraints: List[str]


@dataclass
class EthicalConstraint:
    name: str
    category: str
    description: str
    weight: float
    threshold: float
    check_function: str


@dataclass
class EthicalSolution:
    equipment_config: Dict[str, int]
    overall_utility: float
    stakeholder_utilities: Dict[str, float]
    ethical_compliance: Dict[str, float]
    total_cost: float
    total_productivity: float
    environmental_impact: float
    social_impact: float
    explanation: List[str]


# These illustrative attributes provide the common inputs used by every candidate evaluation.
equipment_types = {
    "RTG": EquipmentType(
        name="Rubber-Tired Gantry",
        capital_cost=2.5,
        operating_cost=0.8,
        productivity=25,
        flexibility=0.8,
        maintenance=0.3,
        footprint=200,
        energy_consumption=3.2,
        environmental_impact=0.6,
        safety_rating=0.75,
        job_creation_factor=2.5,
        community_impact=0.6,
    ),
    "RMG": EquipmentType(
        name="Rail-Mounted Gantry",
        capital_cost=3.2,
        operating_cost=0.6,
        productivity=35,
        flexibility=0.4,
        maintenance=0.25,
        footprint=150,
        energy_consumption=2.8,
        environmental_impact=0.3,
        safety_rating=0.9,
        job_creation_factor=1.2,
        community_impact=0.4,
    ),
    "Straddle": EquipmentType(
        name="Straddle Carrier",
        capital_cost=4.1,
        operating_cost=1.1,
        productivity=20,
        flexibility=0.9,
        maintenance=0.4,
        footprint=100,
        energy_consumption=4.5,
        environmental_impact=0.8,
        safety_rating=0.7,
        job_creation_factor=3.0,
        community_impact=0.8,
    ),
}

print("Equipment types and illustrative attributes:")
for equipment_code, equipment in equipment_types.items():
    print(f"{equipment_code}: {equipment.name}")
    print(f"  Capital cost: ${equipment.capital_cost:.2f}M")
    print(f"  Productivity: {equipment.productivity} containers/hour per unit")
    print(f"  Environmental impact score: {equipment.environmental_impact:.2f}")
    print(f"  Safety rating: {equipment.safety_rating:.2f}")
    print(f"  Estimated jobs per unit: {equipment.job_creation_factor:.1f}")
    print(f"  Community impact score: {equipment.community_impact:.2f}")
    print()

### Value-Aligned Ethical Optimization System

The optimizer uses a finite candidate-generation procedure rather than searching every possible integer configuration. For each candidate, it calculates stakeholder utilities and five compliance scores. The candidate-selection sequence is: generate candidates, calculate metrics, apply budget and productivity feasibility checks, apply the minimum-compliance veto, and compare the weighted selection scores.

A compliance score is a normalized indicator between 0 and 1, but the direction and threshold rule depend on the indicator. Environmental impact is assessed against an upper limit; safety and community impact are assessed against lower limits; estimated job creation is capped at its target; and the equipment-count balance score is compared with its modeled threshold. The stored constraint weights are descriptive inputs and are not used to combine compliance scores. Instead, the selection rule uses the minimum compliance score, so the weakest modeled criterion controls the ethical component and veto.

In [ ]:
class ValueAlignedEthicalOptimizer:
    def __init__(self, equipment_types: Dict[str, EquipmentType]):
        if not equipment_types:
            raise ValueError("At least one equipment type is required.")

        self.equipment_types = equipment_types
        self.stakeholders = []
        self.ethical_constraints = []
        self.optimization_history = []
        self.constitutional_principles = []

        self.ethical_weight = 0.4
        self.economic_weight = 0.3
        self.social_weight = 0.3

        self._initialize_stakeholders()
        self._define_ethical_constraints()
        self._establish_constitutional_principles()

    def _initialize_stakeholders(self):
        """Create the illustrative stakeholder groups and their preferences."""
        stakeholder_configs = [
            {
                "name": "Terminal Operator",
                "category": "Economic",
                "priorities": {
                    "cost_efficiency": 0.4,
                    "productivity": 0.3,
                    "flexibility": 0.2,
                    "maintenance": 0.1,
                },
                "influence": 0.25,
                "utility_function": "economic_utility",
                "ethical_constraints": ["safety_standards", "regulatory_compliance"],
            },
            {
                "name": "Port Authority",
                "category": "Regulatory",
                "priorities": {
                    "safety": 0.4,
                    "environmental_protection": 0.3,
                    "compliance": 0.2,
                    "efficiency": 0.1,
                },
                "influence": 0.20,
                "utility_function": "regulatory_utility",
                "ethical_constraints": ["environmental_limits", "safety_standards", "accessibility"],
            },
            {
                "name": "Local Community",
                "category": "Social",
                "priorities": {
                    "job_creation": 0.35,
                    "environmental_quality": 0.3,
                    "noise_reduction": 0.2,
                    "community_benefits": 0.15,
                },
                "influence": 0.15,
                "utility_function": "social_utility",
                "ethical_constraints": ["environmental_justice", "job_protection", "community_impact"],
            },
            {
                "name": "Environmental Group",
                "category": "Environmental",
                "priorities": {
                    "environmental_impact": 0.5,
                    "sustainability": 0.3,
                    "carbon_footprint": 0.2,
                },
                "influence": 0.20,
                "utility_function": "environmental_utility",
                "ethical_constraints": ["environmental_limits", "sustainability_thresholds", "carbon_neutral"],
            },
            {
                "name": "Labor Union",
                "category": "Social",
                "priorities": {
                    "job_security": 0.4,
                    "safety": 0.3,
                    "fair_wages": 0.2,
                    "working_conditions": 0.1,
                },
                "influence": 0.10,
                "utility_function": "labor_utility",
                "ethical_constraints": ["job_protection", "safety_standards", "fair_labor_practices"],
            },
            {
                "name": "Government Agency",
                "category": "Regulatory",
                "priorities": {
                    "economic_development": 0.3,
                    "environmental_compliance": 0.3,
                    "public_safety": 0.2,
                    "tax_revenue": 0.2,
                },
                "influence": 0.10,
                "utility_function": "government_utility",
                "ethical_constraints": ["regulatory_compliance", "public_interest", "economic_viability"],
            },
        ]

        for config in stakeholder_configs:
            self.stakeholders.append(
                Stakeholder(
                    name=config["name"],
                    category=config["category"],
                    priorities=config["priorities"],
                    influence=config["influence"],
                    utility_function=config["utility_function"],
                    ethical_constraints=config["ethical_constraints"],
                )
            )

        print(f"Initialized {len(self.stakeholders)} stakeholder groups:")
        for stakeholder in self.stakeholders:
            print(
                f"  {stakeholder.name} - {stakeholder.category} "
                f"(influence: {stakeholder.influence:.2f})"
            )

    def _define_ethical_constraints(self):
        """Define threshold-based indicators used in candidate evaluation."""
        self.ethical_constraints = [
            EthicalConstraint(
                name="Environmental Impact Limits",
                category="Environmental",
                description="Average environmental impact should not exceed the modeled limit.",
                weight=0.25,
                threshold=0.7,
                check_function="check_environmental_impact",
            ),
            EthicalConstraint(
                name="Labor Safety Standards",
                category="Social",
                description="Average safety rating should meet the modeled minimum.",
                weight=0.30,
                threshold=0.8,
                check_function="check_labor_safety",
            ),
            EthicalConstraint(
                name="Job Protection Requirements",
                category="Social",
                description="Estimated job creation should meet the modeled minimum.",
                weight=0.20,
                threshold=50,
                check_function="check_job_protection",
            ),
            EthicalConstraint(
                name="Cost Fairness Distribution",
                category="Economic",
                description="The equipment mix should meet the modeled balance score.",
                weight=0.15,
                threshold=0.6,
                check_function="check_cost_fairness",
            ),
            EthicalConstraint(
                name="Community Impact Balance",
                category="Social",
                description="Average community impact should meet the modeled minimum.",
                weight=0.10,
                threshold=0.5,
                check_function="check_community_impact",
            ),
        ]

        print(f"\nDefined {len(self.ethical_constraints)} threshold-based indicators:")
        for constraint in self.ethical_constraints:
            print(
                f"  {constraint.name} ({constraint.category}) "
                f"- descriptive weight: {constraint.weight:.2f}"
            )

    def _establish_constitutional_principles(self):
        """Record principles that frame interpretation of the scoring rules."""
        self.constitutional_principles = [
            {
                "name": "Do No Harm",
                "description": "Avoid choices with a very low modeled compliance score.",
                "priority": 1,
                "application": "minimum-compliance veto",
            },
            {
                "name": "Beneficence",
                "description": "Represent stakeholder wellbeing through positive utility terms.",
                "priority": 2,
                "application": "positive utility contributions",
            },
            {
                "name": "Justice",
                "description": "Include a modeled equipment-mix balance indicator.",
                "priority": 2,
                "application": "balance indicator",
            },
            {
                "name": "Autonomy",
                "description": "Represent preferences as stakeholder-specific priority weights.",
                "priority": 3,
                "application": "preference inputs",
            },
            {
                "name": "Sustainability",
                "description": "Include environmental impact in utilities and compliance.",
                "priority": 2,
                "application": "environmental indicators",
            },
        ]

        print(f"\nRecorded {len(self.constitutional_principles)} guiding principles:")
        for principle in self.constitutional_principles:
            print(f"  {principle['name']} (priority: {principle['priority']})")

    def _validate_equipment_mix(self, equipment_mix: Dict[str, int]):
        """Check that a candidate contains known equipment and nonnegative integer counts."""
        unknown_types = set(equipment_mix) - set(self.equipment_types)
        if unknown_types:
            raise ValueError(f"Unknown equipment types: {sorted(unknown_types)}")
        for equipment_code, count in equipment_mix.items():
            if isinstance(count, bool) or not isinstance(count, (int, np.integer)) or count < 0:
                raise ValueError(
                    f"The count for {equipment_code} must be a nonnegative integer."
                )

    def check_environmental_impact(self, equipment_mix: Dict[str, int]) -> float:
        """Return a compliance score based on the weighted average impact."""
        self._validate_equipment_mix(equipment_mix)
        total_units = sum(equipment_mix.values())
        if total_units == 0:
            return 0.0

        average_impact = sum(
            self.equipment_types[code].environmental_impact * count
            for code, count in equipment_mix.items()
        ) / total_units
        threshold = self.ethical_constraints[0].threshold
        if average_impact <= threshold:
            return 1.0
        return max(0.0, 1.0 - (average_impact - threshold) / (1.0 - threshold))

    def check_labor_safety(self, equipment_mix: Dict[str, int]) -> float:
        """Return a compliance score based on the weighted average safety rating."""
        self._validate_equipment_mix(equipment_mix)
        total_units = sum(equipment_mix.values())
        if total_units == 0:
            return 0.0

        average_safety = sum(
            self.equipment_types[code].safety_rating * count
            for code, count in equipment_mix.items()
        ) / total_units
        threshold = self.ethical_constraints[1].threshold
        if average_safety >= threshold:
            return 1.0
        return max(0.0, average_safety / threshold)

    def check_job_protection(self, equipment_mix: Dict[str, int]) -> float:
        """Return a compliance score based on estimated jobs created."""
        self._validate_equipment_mix(equipment_mix)
        total_jobs = sum(
            self.equipment_types[code].job_creation_factor * count
            for code, count in equipment_mix.items()
        )
        threshold = self.ethical_constraints[2].threshold
        return min(1.0, total_jobs / threshold)

    def check_cost_fairness(self, equipment_mix: Dict[str, int]) -> float:
        """Score how evenly unit counts are distributed across equipment types."""
        self._validate_equipment_mix(equipment_mix)
        total_units = sum(equipment_mix.values())
        if total_units == 0:
            return 0.0

        ideal_distribution = total_units / len(self.equipment_types)
        fairness_score = 1.0
        for equipment_code in self.equipment_types:
            count = equipment_mix.get(equipment_code, 0)
            deviation = abs(count - ideal_distribution)
            fairness_score -= (deviation / ideal_distribution) * 0.2

        fairness_score = max(0.0, fairness_score)
        threshold = self.ethical_constraints[3].threshold
        if fairness_score >= threshold:
            return 1.0
        return max(0.0, fairness_score / threshold)

    def check_community_impact(self, equipment_mix: Dict[str, int]) -> float:
        """Return a compliance score based on weighted average community impact."""
        self._validate_equipment_mix(equipment_mix)
        total_units = sum(equipment_mix.values())
        if total_units == 0:
            return 0.0

        average_impact = sum(
            self.equipment_types[code].community_impact * count
            for code, count in equipment_mix.items()
        ) / total_units
        threshold = self.ethical_constraints[4].threshold
        if average_impact >= threshold:
            return 1.0
        return max(0.0, average_impact / threshold)

    def calculate_stakeholder_utility(
        self,
        equipment_mix: Dict[str, int],
        stakeholder: Stakeholder,
    ) -> float:
        """Calculate utility from the stakeholder's implemented priority terms."""
        self._validate_equipment_mix(equipment_mix)
        total_units = sum(equipment_mix.values())
        if total_units == 0:
            return 0.0

        total_cost = sum(
            equipment_mix.get(code, 0) * equipment.capital_cost
            for code, equipment in self.equipment_types.items()
        )
        total_productivity = sum(
            equipment_mix.get(code, 0) * equipment.productivity
            for code, equipment in self.equipment_types.items()
        )
        average_flexibility = sum(
            equipment_mix.get(code, 0) * equipment.flexibility
            for code, equipment in self.equipment_types.items()
        ) / total_units
        average_environmental_impact = sum(
            equipment_mix.get(code, 0) * equipment.environmental_impact
            for code, equipment in self.equipment_types.items()
        ) / total_units
        average_safety = sum(
            equipment_mix.get(code, 0) * equipment.safety_rating
            for code, equipment in self.equipment_types.items()
        ) / total_units
        total_jobs = sum(
            equipment_mix.get(code, 0) * equipment.job_creation_factor
            for code, equipment in self.equipment_types.items()
        )
        average_community_impact = sum(
            equipment_mix.get(code, 0) * equipment.community_impact
            for code, equipment in self.equipment_types.items()
        ) / total_units
        total_maintenance = sum(
            equipment_mix.get(code, 0) * equipment.maintenance
            for code, equipment in self.equipment_types.items()
        )

        utility = 0.0
        for priority, weight in stakeholder.priorities.items():
            if priority == "cost_efficiency":
                utility += weight / (1.0 + total_cost / 100.0)
            elif priority == "productivity":
                utility += weight * total_productivity / 1000.0
            elif priority == "flexibility":
                utility += weight * average_flexibility
            elif priority == "maintenance":
                utility += weight / (1.0 + total_maintenance / 10.0)
            elif priority == "safety":
                utility += weight * average_safety
            elif priority in {"environmental_protection", "environmental_impact"}:
                utility += weight * (1.0 - average_environmental_impact)
            elif priority in {"job_creation", "job_security"}:
                utility += weight * min(1.0, total_jobs / 100.0)
            elif priority in {"community_benefits", "community_impact"}:
                utility += weight * average_community_impact
            elif priority == "compliance":
                compliance_scores = [
                    self.check_environmental_impact(equipment_mix),
                    self.check_labor_safety(equipment_mix),
                    self.check_job_protection(equipment_mix),
                ]
                utility += weight * float(np.mean(compliance_scores))
            elif priority == "efficiency":
                efficiency_score = total_productivity / (total_cost + 1.0)
                utility += weight * min(1.0, efficiency_score / 50.0)
            # Unimplemented preference terms are deliberately not assigned invented proxies.

        return utility

    def calculate_overall_utility(self, equipment_mix: Dict[str, int]) -> float:
        """Aggregate stakeholder utilities using normalized influence weights."""
        total_influence = sum(stakeholder.influence for stakeholder in self.stakeholders)
        if total_influence <= 0:
            raise ValueError("The total stakeholder influence must be positive.")

        return sum(
            self.calculate_stakeholder_utility(equipment_mix, stakeholder)
            * stakeholder.influence
            / total_influence
            for stakeholder in self.stakeholders
        )

    def calculate_ethical_compliance(
        self,
        equipment_mix: Dict[str, int],
    ) -> Dict[str, float]:
        """Evaluate each named threshold indicator for a candidate configuration."""
        compliance_scores = {}
        for constraint in self.ethical_constraints:
            check_function = getattr(self, constraint.check_function)
            compliance_scores[constraint.name] = check_function(equipment_mix)
        return compliance_scores

    def apply_constitutional_principles(
        self,
        solution: EthicalSolution,
    ) -> EthicalSolution:
        """Record the modeled veto outcome without altering candidate attributes."""
        if solution.ethical_compliance:
            minimum_compliance = min(solution.ethical_compliance.values())
            if minimum_compliance < 0.3:
                solution.explanation.append(
                    "The minimum-compliance veto threshold is not met."
                )
            else:
                solution.explanation.append(
                    "The candidate meets the modeled minimum-compliance veto threshold."
                )
        return solution

    def generate_ethical_solution(
        self,
        equipment_mix: Dict[str, int],
    ) -> EthicalSolution:
        """Calculate the metrics and explanation for one equipment configuration."""
        self._validate_equipment_mix(equipment_mix)
        total_units = sum(equipment_mix.values())
        if total_units == 0:
            return EthicalSolution(
                equipment_config=dict(equipment_mix),
                overall_utility=0.0,
                stakeholder_utilities={},
                ethical_compliance={},
                total_cost=0.0,
                total_productivity=0.0,
                environmental_impact=0.0,
                social_impact=0.0,
                explanation=["No equipment is selected; this is not a usable configuration."],
            )

        total_cost = sum(
            equipment_mix.get(code, 0) * equipment.capital_cost
            for code, equipment in self.equipment_types.items()
        )
        total_productivity = sum(
            equipment_mix.get(code, 0) * equipment.productivity
            for code, equipment in self.equipment_types.items()
        )
        average_environmental_impact = sum(
            equipment_mix.get(code, 0) * equipment.environmental_impact
            for code, equipment in self.equipment_types.items()
        ) / total_units
        total_jobs = sum(
            equipment_mix.get(code, 0) * equipment.job_creation_factor
            for code, equipment in self.equipment_types.items()
        )

        stakeholder_utilities = {
            stakeholder.name: self.calculate_stakeholder_utility(equipment_mix, stakeholder)
            for stakeholder in self.stakeholders
        }
        overall_utility = self.calculate_overall_utility(equipment_mix)
        ethical_compliance = self.calculate_ethical_compliance(equipment_mix)

        explanation = [
            f"Selected candidate configuration: {dict(equipment_mix)}",
            f"Capital cost: ${total_cost:.1f}M; productivity: {total_productivity:.0f} containers/hour.",
            f"Influence-weighted stakeholder utility: {overall_utility:.3f}.",
            "Compliance scores are modeled indicators, not regulatory certifications.",
        ]

        solution = EthicalSolution(
            equipment_config=dict(equipment_mix),
            overall_utility=overall_utility,
            stakeholder_utilities=stakeholder_utilities,
            ethical_compliance=ethical_compliance,
            total_cost=total_cost,
            total_productivity=total_productivity,
            environmental_impact=average_environmental_impact,
            social_impact=total_jobs,
            explanation=explanation,
        )
        return self.apply_constitutional_principles(solution)

    def optimize_equipment_selection_ethical(
        self,
        budget_limit: float = 200,
        min_productivity: float = 500,
    ) -> EthicalSolution:
        """Select the highest-scoring feasible candidate from the generated list."""
        if budget_limit <= 0:
            raise ValueError("The budget limit must be positive.")
        if min_productivity < 0:
            raise ValueError("Minimum productivity cannot be negative.")

        print("=== VALUE-ALIGNED CANDIDATE EVALUATION ===\n")
        print(f"Budget limit: ${budget_limit:.1f}M")
        print(f"Minimum productivity: {min_productivity:.1f} containers/hour\n")

        candidates = self._generate_ethical_candidates(budget_limit)
        best_solution = None
        best_score = -float("inf")

        print(f"Evaluating {len(candidates)} generated candidates...\n")
        for index, candidate in enumerate(candidates, start=1):
            solution = self.generate_ethical_solution(candidate)

            if solution.total_cost > budget_limit or solution.total_productivity < min_productivity:
                continue

            minimum_compliance = min(solution.ethical_compliance.values())
            if minimum_compliance < 0.3:
                continue

            utility_score = solution.overall_utility * self.economic_weight
            ethical_score = minimum_compliance * self.ethical_weight
            efficiency_score = (
                solution.total_productivity / solution.total_cost
            ) * self.social_weight
            final_score = utility_score + ethical_score + efficiency_score

            print(f"Candidate {index}: {candidate}")
            print(
                f"  Utility: {solution.overall_utility:.3f}; "
                f"minimum compliance: {minimum_compliance:.3f}; "
                f"productivity per $M: "
                f"{solution.total_productivity / solution.total_cost:.2f}"
            )
            print(f"  Combined selection score: {final_score:.3f}\n")

            if final_score > best_score:
                best_score = final_score
                best_solution = solution

        self.optimization_history.append(
            {
                "budget_limit": budget_limit,
                "min_productivity": min_productivity,
                "candidate_count": len(candidates),
                "best_score": best_score if best_solution is not None else None,
            }
        )
        return best_solution

    def _generate_ethical_candidates(
        self,
        budget_limit: float,
    ) -> List[Dict[str, int]]:
        """Scale the stated base mixes and retain the generated integer candidates."""
        base_configs = [
            {"RTG": 12, "RMG": 8, "Straddle": 10},
            {"RTG": 15, "RMG": 6, "Straddle": 9},
            {"RTG": 8, "RMG": 12, "Straddle": 10},
            {"RTG": 10, "RMG": 7, "Straddle": 13},
            {"RTG": 14, "RMG": 10, "Straddle": 6},
            {"RTG": 6, "RMG": 8, "Straddle": 16},
            {"RTG": 11, "RMG": 9, "Straddle": 10},
            {"RTG": 9, "RMG": 11, "Straddle": 10},
        ]

        candidates = []
        for config in base_configs:
            base_cost = sum(
                count * self.equipment_types[code].capital_cost
                for code, count in config.items()
            )
            if base_cost <= budget_limit * 1.1:
                scale_factor = budget_limit / base_cost
                scaled_config = {
                    code: max(1, int(count * scale_factor))
                    for code, count in config.items()
                }
                candidates.append(scaled_config)

        return candidates


print("Value-aligned candidate-scoring system is ready.")

### Initialize the Ethical Optimization System

Influence values are normalized when stakeholder utilities are aggregated. The economic, ethical, and social weights are used later in the combined candidate score; they do not alter the stakeholder-level utility calculations.

In [ ]:
ethical_optimizer = ValueAlignedEthicalOptimizer(equipment_types)

print("\n=== SYSTEM INITIALIZATION COMPLETE ===")
print(f"Stakeholder groups: {len(ethical_optimizer.stakeholders)}")
print(f"Compliance indicators: {len(ethical_optimizer.ethical_constraints)}")
print(f"Guiding principles: {len(ethical_optimizer.constitutional_principles)}")

total_influence = sum(stakeholder.influence for stakeholder in ethical_optimizer.stakeholders)
print("\nNormalized stakeholder influence:")
for stakeholder in ethical_optimizer.stakeholders:
    influence_percentage = stakeholder.influence / total_influence * 100
    print(f"  {stakeholder.name}: {influence_percentage:.1f}%")

print("\nCombined-score weights:")
print(f"  Economic utility: {ethical_optimizer.economic_weight:.1%}")
print(f"  Minimum compliance: {ethical_optimizer.ethical_weight:.1%}")
print(f"  Productivity per capital cost: {ethical_optimizer.social_weight:.1%}")

### Run Value-Aligned Candidate Evaluation

The generated configurations are evaluated in their defined order. A candidate must meet the budget and productivity requirements and pass the minimum-compliance veto before its combined score can be considered. The label “selected candidate” is precise here: the finite generated list is not an exhaustive search of all possible equipment mixes.

In [ ]:
print("=== CANDIDATE EVALUATION ===\n")

budget_limit = 200.0
min_productivity = 500.0

best_solution = ethical_optimizer.optimize_equipment_selection_ethical(
    budget_limit=budget_limit,
    min_productivity=min_productivity,
)

if best_solution is not None:
    print("\n=== HIGHEST-SCORING FEASIBLE CANDIDATE ===\n")
    print("Equipment configuration:")
    for equipment_code, count in best_solution.equipment_config.items():
        print(f"  {equipment_code}: {count} units")

    print("\nPerformance metrics:")
    print(f"  Capital cost: ${best_solution.total_cost:.1f}M")
    print(f"  Productivity: {best_solution.total_productivity:.0f} containers/hour")
    print(
        f"  Productivity per capital cost: "
        f"{best_solution.total_productivity / best_solution.total_cost:.2f} containers/hour per $M"
    )
    print(f"  Budget utilization: {best_solution.total_cost / budget_limit:.1%}")
    print(f"\nInfluence-weighted stakeholder utility: {best_solution.overall_utility:.3f}")

    print("\nStakeholder utilities:")
    for stakeholder_name, utility in best_solution.stakeholder_utilities.items():
        print(f"  {stakeholder_name}: {utility:.3f}")

    print("\nModeled compliance scores:")
    for constraint_name, compliance in best_solution.ethical_compliance.items():
        print(f"  {constraint_name}: {compliance:.3f}")

    average_compliance = float(np.mean(list(best_solution.ethical_compliance.values())))
    print(f"\nAverage modeled compliance score: {average_compliance:.1%}")
    print("\nImpact indicators:")
    print(
        f"  Average environmental impact score: "
        f"{best_solution.environmental_impact:.3f} (lower is better)"
    )
    print(f"  Estimated jobs: {best_solution.social_impact:.1f}")

    print("\nDecision notes:")
    for note_number, explanation in enumerate(best_solution.explanation, start=1):
        print(f"  {note_number}. {explanation}")
else:
    print("No generated candidate meets all modeled feasibility and veto rules.")

### Stakeholder Analysis and Trade-Off Visualization

The panels show stakeholder utilities, compliance scores, equipment-level contributions, and a comparison with a fixed illustrative configuration. In the equipment panel, each bar is an attribute multiplied by unit count; these attributes have different units and should be compared within each attribute, not directly against one another. The trade-off panel normalizes productivity per capital cost by 50 so that it can be displayed alongside 0–1 indicators; this display normalization does not change candidate selection.

In [ ]:
if best_solution is not None:
    fig = plt.figure(figsize=(16, 12))
    fig.suptitle("Stakeholder and Modeled Impact Analysis", fontsize=16, fontweight="bold")
    ax1 = fig.add_subplot(2, 2, 1)
    ax2 = fig.add_subplot(2, 2, 2, projection="polar")
    ax3 = fig.add_subplot(2, 2, 3)
    ax4 = fig.add_subplot(2, 2, 4)

    stakeholder_names = list(best_solution.stakeholder_utilities)
    utility_values = list(best_solution.stakeholder_utilities.values())
    utility_colors = [
        "green" if value >= 0.7 else "orange" if value >= 0.5 else "red"
        for value in utility_values
    ]
    bars = ax1.bar(stakeholder_names, utility_values, color=utility_colors, alpha=0.75)
    ax1.set_xlabel("Stakeholder group")
    ax1.set_ylabel("Utility score")
    ax1.set_title("Stakeholder Utility")
    ax1.tick_params(axis="x", rotation=45)
    ax1.set_ylim(0, 1)
    ax1.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, utility_values):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    constraint_names = list(best_solution.ethical_compliance)
    compliance_values = list(best_solution.ethical_compliance.values())
    angles = np.linspace(0, 2 * np.pi, len(constraint_names), endpoint=False)
    closed_angles = np.append(angles, angles[0])
    closed_values = compliance_values + compliance_values[:1]
    ax2.plot(closed_angles, closed_values, "o-", linewidth=2, color="blue")
    ax2.fill(closed_angles, closed_values, alpha=0.25, color="blue")
    ax2.set_xticks(angles)
    ax2.set_xticklabels(constraint_names, fontsize=8)
    ax2.set_ylim(0, 1)
    ax2.set_title("Modeled Compliance Profile", pad=20)
    ax2.grid(True)

    equipment_names = list(best_solution.equipment_config)
    equipment_counts = [best_solution.equipment_config[code] for code in equipment_names]
    environmental_contributions = [
        equipment_types[code].environmental_impact * count
        for code, count in zip(equipment_names, equipment_counts)
    ]
    safety_contributions = [
        equipment_types[code].safety_rating * count
        for code, count in zip(equipment_names, equipment_counts)
    ]
    job_contributions = [
        equipment_types[code].job_creation_factor * count
        for code, count in zip(equipment_names, equipment_counts)
    ]

    positions = np.arange(len(equipment_names))
    bar_width = 0.25
    ax3.bar(
        positions - bar_width,
        environmental_contributions,
        bar_width,
        label="Environmental impact score × units",
        color="red",
        alpha=0.7,
    )
    ax3.bar(
        positions,
        safety_contributions,
        bar_width,
        label="Safety rating × units",
        color="green",
        alpha=0.7,
    )
    ax3.bar(
        positions + bar_width,
        job_contributions,
        bar_width,
        label="Estimated jobs",
        color="blue",
        alpha=0.7,
    )
    ax3.set_xlabel("Equipment type")
    ax3.set_ylabel("Attribute contribution (attribute-specific units)")
    ax3.set_title("Equipment Attribute Contributions")
    ax3.set_xticks(positions)
    ax3.set_xticklabels(equipment_names)
    ax3.legend(fontsize=8)
    ax3.grid(axis="y", alpha=0.3)

    benchmark_config = {"RTG": 8, "RMG": 15, "Straddle": 6}
    benchmark_solution = ethical_optimizer.generate_ethical_solution(benchmark_config)
    objective_names = [
        "Productivity per cost\n(normalized)",
        "Average compliance",
        "Estimated jobs\n(normalized)",
        "Environmental\nsustainability",
    ]
    selected_scores = [
        min(1.0, best_solution.total_productivity / best_solution.total_cost / 50.0),
        float(np.mean(list(best_solution.ethical_compliance.values()))),
        min(1.0, best_solution.social_impact / 100.0),
        1.0 - best_solution.environmental_impact,
    ]
    benchmark_scores = [
        min(1.0, benchmark_solution.total_productivity / benchmark_solution.total_cost / 50.0),
        float(np.mean(list(benchmark_solution.ethical_compliance.values()))),
        min(1.0, benchmark_solution.social_impact / 100.0),
        1.0 - benchmark_solution.environmental_impact,
    ]

    objective_positions = np.arange(len(objective_names))
    comparison_width = 0.35
    ax4.bar(
        objective_positions - comparison_width / 2,
        selected_scores,
        comparison_width,
        label="Selected candidate",
        color="blue",
        alpha=0.75,
    )
    ax4.bar(
        objective_positions + comparison_width / 2,
        benchmark_scores,
        comparison_width,
        label="Illustrative benchmark configuration",
        color="red",
        alpha=0.65,
    )
    ax4.set_xlabel("Displayed indicator")
    ax4.set_ylabel("Normalized score")
    ax4.set_title("Indicator Comparison (not a second optimization)")
    ax4.set_xticks(objective_positions)
    ax4.set_xticklabels(objective_names, fontsize=8)
    ax4.set_ylim(0, 1)
    ax4.legend(fontsize=8)
    ax4.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()
    print("The charts summarize the model indicators for the selected and benchmark configurations.")

### Ethical Decision Quality Assessment

These summary measures describe the selected candidate under the notebook's scoring conventions. Utility variance is calculated across stakeholder groups without influence weighting, while overall utility uses influence weights. Consequently, the two measures answer different questions. The quality score is a constructed indicator, not a validated ethical standard; its equity component can be negative if utility variance exceeds 1.

In [ ]:
if best_solution is not None:
    print("=== MODELED DECISION-QUALITY INDICATORS ===\n")

    utility_values = list(best_solution.stakeholder_utilities.values())
    compliance_values = list(best_solution.ethical_compliance.values())
    average_utility = float(np.mean(utility_values))
    minimum_utility = min(utility_values)
    utility_variance = float(np.var(utility_values))
    average_compliance = float(np.mean(compliance_values))
    minimum_compliance = min(compliance_values)

    satisfied_stakeholders = sum(value >= 0.7 for value in utility_values)
    total_stakeholders = len(utility_values)
    satisfaction_rate = satisfied_stakeholders / total_stakeholders
    satisfied_constraints = sum(value >= 0.7 for value in compliance_values)
    total_constraints = len(compliance_values)
    constraint_satisfaction_rate = satisfied_constraints / total_constraints

    print("STAKEHOLDER INDICATORS:")
    print(f"  Unweighted average utility: {average_utility:.3f}")
    print(f"  Minimum stakeholder utility: {minimum_utility:.3f}")
    print(f"  Utility variance: {utility_variance:.3f} (lower indicates more similar scores)")
    print(
        f"  Share with utility at least 0.7: {satisfaction_rate:.1%} "
        f"({satisfied_stakeholders}/{total_stakeholders})\n"
    )

    print("COMPLIANCE INDICATORS:")
    print(f"  Average compliance score: {average_compliance:.3f}")
    print(f"  Minimum compliance score: {minimum_compliance:.3f}")
    print(
        f"  Share of indicators at least 0.7: {constraint_satisfaction_rate:.1%} "
        f"({satisfied_constraints}/{total_constraints})\n"
    )

    utility_quality = average_utility * 0.3
    equity_quality = (1.0 - utility_variance) * 0.2
    compliance_quality = average_compliance * 0.3
    robustness_quality = minimum_compliance * 0.2
    overall_quality = (
        utility_quality + equity_quality + compliance_quality + robustness_quality
    )

    print("CONSTRUCTED QUALITY INDICATOR:")
    print(f"  Utility component: {utility_quality:.3f}")
    print(f"  Equity component: {equity_quality:.3f}")
    print(f"  Compliance component: {compliance_quality:.3f}")
    print(f"  Minimum-compliance component: {robustness_quality:.3f}")
    print(f"  Combined indicator: {overall_quality:.3f}\n")

    if overall_quality >= 0.9:
        quality_level = "HIGH UNDER THIS SCORING RULE"
    elif overall_quality >= 0.8:
        quality_level = "FAVORABLE UNDER THIS SCORING RULE"
    elif overall_quality >= 0.7:
        quality_level = "MODERATE UNDER THIS SCORING RULE"
    else:
        quality_level = "LOW UNDER THIS SCORING RULE"
    print(f"Classification: {quality_level}\n")

    print("MODELED AREAS FOR REVIEW:")
    improvement_areas = []
    if minimum_utility < 0.5:
        low_utility_stakeholders = [
            name
            for name, value in best_solution.stakeholder_utilities.items()
            if value < 0.5
        ]
        improvement_areas.append(
            f"Stakeholder utility below 0.5: {', '.join(low_utility_stakeholders)}"
        )
    if minimum_compliance < 0.5:
        low_compliance_indicators = [
            name
            for name, value in best_solution.ethical_compliance.items()
            if value < 0.5
        ]
        improvement_areas.append(
            f"Compliance score below 0.5: {', '.join(low_compliance_indicators)}"
        )
    if utility_variance > 0.1:
        improvement_areas.append(
            "Stakeholder utilities differ substantially under the unweighted variance measure."
        )

    if improvement_areas:
        for area in improvement_areas:
            print(f"  - {area}")
    else:
        print("  No review trigger in these selected indicators.")

    print(
        "\nInterpretation: these indicators can guide discussion, but they do not establish "
        "stakeholder approval, legal compliance, or implementation readiness."
    )

### Results Summary and Implementation Guidance

The final summary distinguishes calculated model outputs from judgments that require external evidence. The benchmark in the visualization is one specified configuration; it is not independently optimized and does not represent a pure economic optimum. Use the model to inspect assumptions and trade-offs, then validate data, constraints, and stakeholder preferences before making a real investment decision.

In [ ]:
print("=== VALUE-ALIGNED EQUIPMENT SELECTION: SUMMARY ===\n")
print("Problem: equipment selection with modeled stakeholder preferences and indicators")
print("Method: finite candidate generation, feasibility checks, veto, and weighted scoring\n")

if best_solution is not None:
    print("MODEL SETUP:")
    print(f"  Stakeholder groups: {len(ethical_optimizer.stakeholders)}")
    print(f"  Compliance indicators: {len(ethical_optimizer.ethical_constraints)}")
    print(f"  Guiding principles: {len(ethical_optimizer.constitutional_principles)}")
    print(f"  Budget limit: ${budget_limit:.1f}M")
    print(f"  Minimum productivity: {min_productivity:.1f} containers/hour\n")

    print("SELECTED GENERATED CANDIDATE:")
    for equipment_code, count in best_solution.equipment_config.items():
        print(f"  {equipment_code}: {count} units")
    print(f"  Capital cost: ${best_solution.total_cost:.1f}M")
    print(f"  Productivity: {best_solution.total_productivity:.0f} containers/hour")
    print(
        f"  Productivity per capital cost: "
        f"{best_solution.total_productivity / best_solution.total_cost:.2f} containers/hour per $M\n"
    )

    print("STAKEHOLDER UTILITY SCORES:")
    for stakeholder_name, utility in best_solution.stakeholder_utilities.items():
        level = "higher" if utility >= 0.7 else "middle" if utility >= 0.5 else "lower"
        print(f"  {stakeholder_name}: {utility:.3f} ({level} under the model scale)")
    print(f"  Influence-weighted utility: {best_solution.overall_utility:.3f}\n")

    print("MODELED COMPLIANCE SCORES:")
    for constraint_name, compliance in best_solution.ethical_compliance.items():
        print(f"  {constraint_name}: {compliance:.3f}")
    print(f"  Mean compliance score: {average_compliance:.1%}\n")

    print("SOCIAL AND ENVIRONMENTAL INDICATORS:")
    print(f"  Estimated jobs: {best_solution.social_impact:.1f}")
    print(
        f"  Average environmental impact score: "
        f"{best_solution.environmental_impact:.3f} (lower is better)\n"
    )

    print("CONSTRUCTED QUALITY INDICATOR:")
    print(f"  Combined indicator: {overall_quality:.3f}")
    print(f"  Classification: {quality_level}")
    print(f"  Stakeholder utility threshold share: {satisfaction_rate:.1%}")
    print(f"  Compliance indicator threshold share: {constraint_satisfaction_rate:.1%}\n")

    print("HOW THE GUIDING PRINCIPLES APPEAR IN THIS MODEL:")
    print("  Do No Harm: a minimum-compliance veto is applied at 0.3.")
    print("  Beneficence: implemented stakeholder utility terms contribute to the score.")
    print("  Justice: an equipment-count balance indicator is included.")
    print("  Autonomy: stakeholder priorities are represented as input weights.")
    print("  Sustainability: environmental attributes contribute to utility and compliance.\n")

    if "benchmark_solution" in globals():
        print("FIXED BENCHMARK CONFIGURATION:")
        print(f"  Configuration: {benchmark_solution.equipment_config}")
        print(
            "  It is a reference mix for the plotted indicator comparison, "
            "not an optimized economic solution.\n"
        )

    print("INTERPRETATION AND PRACTICAL USE:")
    print("  - The selected mix is the highest-scoring candidate in the generated list.")
    print("  - Scores depend on illustrative attributes and implemented utility terms.")
    print("  - Several stakeholder priorities have no implemented utility calculation.")
    print("  - The compliance scores are not legal certifications or measured ethical outcomes.")
    print("  - A real assessment requires data validation and meaningful stakeholder engagement.")
    print("  - Any implementation decision requires independent operational and regulatory review.")
else:
    print("No generated candidate meets the budget, productivity, and modeled veto rules.")
    print("The candidate list and assumptions define the scope of this result.")

print("\n=== CANDIDATE EVALUATION COMPLETE ===")